# Decaying activation score (research prototype)

A running score that remembers recent large stimuli but fades over time. At each
step the new score is the best (largest) of three possibilities: the previous
score reduced by 2, the score from two steps back reduced by 1, or a fresh stimulus
value for this step. Because the two-steps-back term fades more slowly than the
one-step-back term, the score is governed by a two-step memory, not just the last
value. The stimulus uses a fixed deterministic fill so the result is reproducible;
we report the checksum (sum of all scores) as the oracle. The C port must reproduce
this checksum.

In [ ]:
N_TEST = 1000000

# per-step gains (integers): the one-step-back term loses 2, the two-steps-back term
# loses 1, so the slower-fading two-step memory governs the decay
A = -2
B = -1

# a score can never be less than the sentinel; the first two steps have no history, so
# they see the sentinel for the missing earlier scores
NEG = -10**9

# deterministic stimulus in the integer range [0, 1000); exact integer arithmetic, so a
# C port using unsigned 64-bit arithmetic reproduces every value bit-for-bit
def stim_of(i):
    return (i * 2654435761) % 1000003 % 1000

w = [stim_of(i) for i in range(N_TEST)]

In [ ]:
# the score at step i depends on the two previous scores (inherently sequential as
# written): out[i] = max(A + out[i-1], B + out[i-2], w[i]); missing earlier scores are
# the sentinel, so out[0] = w[0] and out[1] = max(A + out[0], w[1]).
def decay(w):
    n = len(w)
    out = [0] * n
    prev1 = NEG   # out[i-1]
    prev2 = NEG   # out[i-2]
    for i in range(n):
        x = max(A + prev1, B + prev2, w[i])
        out[i] = x
        prev2 = prev1
        prev1 = x
    return out

out = decay(w)
checksum = 0
for v in out:
    checksum += v
print('checksum =', checksum)

In [ ]:
# The oracle: this notebook's own output. A port of the algorithm (in any language)
# must reproduce this checksum within tol. The scores are all integers and stay well
# below 2**53, so the checksum is exact in double precision and any correct port --
# serial or parallel, however it splits the work -- must match it to well under 1. The
# scores are handed over as the terms of the checksum so the extraction tool can
# confirm the sum reproduces the expected value and measure its (here zero) spread
# under reordering.
ORACLE = {'kernel': 'decay',
          'params': {'N': N_TEST},
          'expected': {'CHECKSUM': float(checksum)},
          'terms': {'CHECKSUM': [float(v) for v in out]},
          'tol': 0.5}